In [23]:
import json
import cv2
import random
from pathlib import Path
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image

%matplotlib inline


## 0.1 Paths

`pvsg.json` (3.88MB) can be downloaded directly from
https://huggingface.co/datasets/Jingkang/PVSG/resolve/main/pvsg.json --
no need to clone the whole repo or download the 10.8GB of masks/videos just
for annotation-level exploration.

Mirroring OpenPVSG's own expected structure (so any of their scripts still
work unmodified if you use them later), assumed layout:

```
master-degree-project/
└── data/PVSG_Dataset/
    ├── pvsg.json
    └── data/
        ├── vidor/{frames, masks, videos}
        ├── epic_kitchen/{frames, masks, videos}
        └── ego4d/{frames, masks, videos}
```

Adjust `PROJECT_ROOT` below if this notebook doesn't sit at
`src/notebooks/`, same as the ASPIRe notebook.


In [114]:
PROJECT_ROOT = Path.cwd().resolve().parents[1]

VIDOR_ROOT = PROJECT_ROOT / "data" / "ViDOR"
VIDEO_TRAIN_DATA = VIDOR_ROOT / "train" / "video"
TRAIN_DATA_JSON = VIDOR_ROOT / "train_files.json"
TRAIN_DATA_ANNOTATIONS = VIDOR_ROOT / "train" / "training_annotation"  # frames/masks/videos per source live under here

print("PROJECT_ROOT   :", PROJECT_ROOT, "-> exists:", PROJECT_ROOT.exists())
print("VIDOR_ROOT      :", VIDOR_ROOT, "-> exists:", VIDOR_ROOT.exists())
print("VIDEO_TRAIN_DATA      :", VIDEO_TRAIN_DATA, "-> exists:", VIDEO_TRAIN_DATA.exists())
print("TRAIN_DATA :", TRAIN_DATA_JSON, "-> exists:", TRAIN_DATA_JSON.exists())
print("TRAIN_DATA_ANNOTATIONS :", TRAIN_DATA_ANNOTATIONS, "-> exists:", TRAIN_DATA_ANNOTATIONS.exists())

# assert PVSG_JSON.exists(), (
#     "pvsg.json not found -- download it from "
#     "https://huggingface.co/datasets/Jingkang/PVSG/resolve/main/pvsg.json "
#     "and place it at the PVSG_JSON path above before continuing."
# )


PROJECT_ROOT   : C:\Users\Samuel Oliveira\Desktop\CS\master-degree-project -> exists: True
VIDOR_ROOT      : C:\Users\Samuel Oliveira\Desktop\CS\master-degree-project\data\ViDOR -> exists: True
VIDEO_TRAIN_DATA      : C:\Users\Samuel Oliveira\Desktop\CS\master-degree-project\data\ViDOR\train\video -> exists: True
TRAIN_DATA : C:\Users\Samuel Oliveira\Desktop\CS\master-degree-project\data\ViDOR\train_files.json -> exists: True
TRAIN_DATA_ANNOTATIONS : C:\Users\Samuel Oliveira\Desktop\CS\master-degree-project\data\ViDOR\train\training_annotation -> exists: True


In [115]:
OUTPUT_PATH = PROJECT_ROOT / "outputs" / "notebooks" / "20260725_ViDOR_dataset_exploration"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

In [116]:
video_code = "6001671251"
video_idx = "1008"

output_video = OUTPUT_PATH / f"path_{video_idx}_code_{video_code}.mp4"

In [117]:
path_to_video_annotation = TRAIN_DATA_ANNOTATIONS / f"{video_idx}" / f"{video_code}.json"
video_path = VIDEO_TRAIN_DATA / f"{video_idx}" / f"{video_code}.mp4"

with open(path_to_video_annotation, "r") as f:
    annotation = json.load(f)

print(f"Video ID     : {annotation['video_id']}")
print(f"Frame count  : {annotation['frame_count']}")
print(f"FPS          : {annotation['fps']}")
print(f"Resolution   : {annotation['width']} x {annotation['height']}")
print()

# tid -> category
tid_to_category = {
    obj["tid"]: obj["category"]
    for obj in annotation["subject/objects"]
}

trajectories = annotation["trajectories"]
relations = annotation["relation_instances"]

print("Objects in video:")
for tid, category in tid_to_category.items():
    print(f"  tid={tid}: {category}")




Video ID     : 6001671251
Frame count  : 1180
FPS          : 29.97002997002997
Resolution   : 640 x 360

Objects in video:
  tid=0: adult
  tid=1: adult
  tid=2: adult
  tid=3: adult
  tid=4: adult
  tid=5: adult


In [118]:
video_path

WindowsPath('C:/Users/Samuel Oliveira/Desktop/CS/master-degree-project/data/ViDOR/train/video/1008/6001671251.mp4')

# Object + Relations Tracking

In [119]:
# ==============================================================================
# Video
# ==============================================================================

cap = cv2.VideoCapture(str(video_path))

if not cap.isOpened():
    raise RuntimeError("Cannot open video.")

frame_idx = 0

# Get video properties
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# MP4 codec
fourcc = cv2.VideoWriter_fourcc(*"mp4v")

writer = cv2.VideoWriter(
    str(output_video),
    fourcc,
    fps,
    (width, height),
)


In [120]:
def _tid(x):
    """Normalize a track id to plain int, so dict keys/lookups always match
    regardless of whether the source JSON gave us int or str tids."""
    return int(x)

In [121]:
# ==============================================================================
# Main loop
# ==============================================================================

frame_idx = 0
total_active_relation_instances = 0  # running diagnostic across the whole video
frames_with_zero_active_relations = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_annotations = trajectories[frame_idx]

    # -------------------------------------------------------------------------
    # Find active relations for this frame
    # -------------------------------------------------------------------------

    active_relations = []

    for rel in relations:

        if rel["begin_fid"] <= frame_idx < rel["end_fid"]:
            active_relations.append(rel)

    total_active_relation_instances += len(active_relations)
    if len(active_relations) == 0:
        frames_with_zero_active_relations += 1

    # -------------------------------------------------------------------------
    # Group relations by SUBJECT
    #
    # Example:
    #
    # person (0)
    #     next_to, behind -> chair (1)
    #     towards -> toy (2)
    #
    # FIX (Bug 1): cast subject_tid/object_tid through _tid() so the keys
    # here are guaranteed to be plain int, matching how we'll look them up
    # below via _tid(obj["tid"]). Previously, if subject_tid came through as
    # e.g. a numpy int64, a str, or any type that doesn't hash/compare equal
    # to a plain Python int, "if tid in relations_by_subject" would silently
    # always be False -- no error, relation text just never renders.
    # -------------------------------------------------------------------------

    relations_by_subject = defaultdict(lambda: defaultdict(list))

    for rel in active_relations:

        s_tid = _tid(rel["subject_tid"])
        o_tid = _tid(rel["object_tid"])

        predicate = rel["predicate"]
        object_name = tid_to_category[o_tid]

        relations_by_subject[s_tid][f"{object_name} ({o_tid})"].append(predicate)

    # -------------------------------------------------------------------------
    # Draw objects
    # -------------------------------------------------------------------------

    for obj in frame_annotations:

        tid = _tid(obj["tid"])  # FIX (Bug 1): same normalization on lookup side

        category = tid_to_category[tid]

        bbox = obj["bbox"]

        xmin = int(bbox["xmin"])
        ymin = int(bbox["ymin"])
        xmax = int(bbox["xmax"])
        ymax = int(bbox["ymax"])

        generated = obj["generated"]

        color = (0, 255, 0) if generated == 0 else (0, 255, 255)

        # -------------------------------------------------------------
        # Bounding box
        # -------------------------------------------------------------

        cv2.rectangle(
            frame,
            (xmin, ymin),
            (xmax, ymax),
            color,
            2,
        )

        # -------------------------------------------------------------
        # Object name
        # -------------------------------------------------------------

        cv2.putText(
            frame,
            f"{category} ({tid})",
            (xmin, max(ymin - 5, 15)),  # FIX (Bug 2): clamp so label never
            cv2.FONT_HERSHEY_SIMPLEX,   # goes above the visible frame either
            0.60,
            color,
            2,
            cv2.LINE_AA,
        )

        # -------------------------------------------------------------
        # Relations
        # -------------------------------------------------------------

        if tid in relations_by_subject:

            relation_lines = []

            for target, predicates in relations_by_subject[tid].items():

                text = f"{', '.join(predicates)} -> {target}"

                relation_lines.append(text)

            # FIX (Bug 2): the stack of relation lines was previously
            # positioned purely relative to ymin, with no floor -- for any
            # object near the top of the frame, or with several stacked
            # relations, `y` went negative (off-screen) and cv2.putText
            # drew nothing visible. Clamp the starting y so the whole stack
            # always fits on-screen, growing downward from a safe minimum
            # instead of upward off the top.
            block_height = 18 * len(relation_lines)
            y = max(ymin - 25 - block_height, 15 + block_height)
            y -= block_height  # start of the block, so the loop below fills downward correctly

            for line in relation_lines:

                # white background rectangle

                (w, h), baseline = cv2.getTextSize(
                    line,
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.45,
                    1,
                )

                cv2.rectangle(
                    frame,
                    (xmin - 2, y - h - 2),
                    (xmin + w + 2, y + baseline),
                    (255, 255, 255),
                    -1,
                )

                cv2.putText(
                    frame,
                    line,
                    (xmin, y),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.45,
                    (0, 0, 0),
                    1,
                    cv2.LINE_AA,
                )

                y += 18

    # -------------------------------------------------------------------------
    # Frame number + live diagnostic counter
    #
    # NEW: always-visible "active relations: N" readout. If this reads 0 for
    # every frame throughout the whole video, the bug is upstream (relations
    # not loaded correctly, or begin_fid/end_fid never matching frame_idx --
    # e.g. an off-by-one or frame-rate mismatch between the video file and
    # the annotation). If it reads >0 but you still don't see text, the bug
    # is in positioning/rendering, not matching.
    # -------------------------------------------------------------------------

    cv2.putText(
        frame,
        f"Frame: {frame_idx}",
        (10, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 0, 255),
        2,
    )

    cv2.putText(
        frame,
        f"Active relations: {len(active_relations)}",
        (10, 65),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 0, 255),
        2,
    )

    # -------------------------------------------------------------------------
    # Show
    # -------------------------------------------------------------------------
    writer.write(frame)

    cv2.imshow("VIDOR Scene Graph", frame)

    key = cv2.waitKey(30) & 0xFF

    if key == 27:      # ESC
        break

    elif key == ord(" "):  # Pause
        cv2.waitKey(0)

    frame_idx += 1

cap.release()
writer.release()
cv2.destroyAllWindows()

# -------------------------------------------------------------------------
# End-of-run diagnostic summary -- tells you at a glance whether relations
# were found at all across the whole video, without having to scrub through
# the output looking for the on-screen counter.
# -------------------------------------------------------------------------
print(f"Total (frame, relation) active instances summed across the video: {total_active_relation_instances}")
print(f"Frames with zero active relations: {frames_with_zero_active_relations} / {frame_idx}")
if total_active_relation_instances == 0:
    print("-> No active relations were ever found. This points to an upstream "
          "problem (relations list empty/mis-parsed, or begin_fid/end_fid never "
          "overlapping frame_idx), not a rendering bug. Check how `relations` "
          "and `trajectories` were loaded, and confirm begin_fid/end_fid use the "
          "same 0-indexed, frame-count convention as `frame_idx` here.")

print(f"Saved visualization to:\n{output_video}")

Total (frame, relation) active instances summed across the video: 20128
Frames with zero active relations: 10 / 1180
Saved visualization to:
C:\Users\Samuel Oliveira\Desktop\CS\master-degree-project\outputs\notebooks\20260725_ViDOR_dataset_exploration\path_1008_code_6001671251.mp4
